# Day 02 — preprocessing финансовых предложений

## D2-01 — контракт минимальной очистки и setup

**Цель:** определить policy `minimal-v1` и загрузить реальные исходные данные.
Вход — локальный canonical CSV Financial PhraseBank `sentences_75agree`;
на этом checkpoint выход — исходный DataFrame и согласованный контракт.
Очистка появится в D2-02, обработка таблицы — в D2-03.

**Краткая теория.** Preprocessing преобразует представление текста.
Удаление `not` меняет отрицание, удаление `-5%` теряет направление и величину
изменения. Поэтому сохраняем эти сигналы. Детерминированная очистка не требует
`fit`; TF-IDF изучает vocabulary/IDF и позже будет обучаться только на train.

Источник: takala/financial_phrasebank; Malo et al. (2014); CC-BY-NC-SA-3.0.
Если локального CSV нет, подготовьте его отдельной командой из корня:
`PYTHONPATH=src venv/bin/python -m finnews_sentiment.data.prepare_phrasebank --download`.
Notebook не скачивает данные.

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "SPEC.md").is_file() and (candidate / "src/finnews_sentiment").is_dir():
            return candidate
    raise FileNotFoundError("Project root with SPEC.md and src/finnews_sentiment not found")


ROOT: Path = find_project_root(Path.cwd().resolve())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from finnews_sentiment.data.load_data import CLASS_ORDER, load_raw_news

print(f"Interpreter: {sys.executable}")
print(f"Project root: {ROOT}")
print(f"Class order: {CLASS_ORDER}")

Interpreter: /home/makarlistkov/projects/Text-classification/venv/bin/python
Project root: /home/makarlistkov/projects/Text-classification
Class order: ('negative', 'neutral', 'positive')


### Вызов кода: загрузка canonical CSV

Проверяем файл до/после чтения и совпадение повторной загрузки.
Это проверка сохранности raw representation, а не применение preprocessing.
На checkpoint D2-01 `features.preprocess` не импортировался: тогда это была
непринятая заготовка с NLTK resource access при import. Ниже, в D2-02,
импортируется уже проверенная реализация очистки одного текста.

In [2]:
import hashlib
import pandas as pd
from IPython.display import display

CANONICAL_PATH = ROOT / "data/interim/financial_phrasebank_75agree.csv"
if not CANONICAL_PATH.is_file():
    raise FileNotFoundError("Canonical CSV is missing; run the explicit preparation command above")

canonical_sha256_before = hashlib.sha256(CANONICAL_PATH.read_bytes()).hexdigest()
raw_news = load_raw_news(CANONICAL_PATH)
raw_snapshot = raw_news.copy(deep=True)
print(f"Raw DataFrame shape: {raw_news.shape}")
print(f"Canonical CSV SHA-256: {canonical_sha256_before}")
display(raw_news.head())
display(raw_news["sentiment"].value_counts().reindex(CLASS_ORDER, fill_value=0).rename("count").to_frame())

Raw DataFrame shape: (3453, 2)
Canonical CSV SHA-256: 791b78a0be391241bd6ea600fdd0128caee7c0dbe89eed10dfd00de505fb3c2e


,text,sentiment
0,"According to Gran , the company has no plans t...",neutral
1,With the new production plant the company woul...,positive
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,"In the third quarter of 2010 , net sales incre...",positive
4,Operating profit rose to EUR 13.1 mn from EUR ...,positive


,count
sentiment,
negative,420
neutral,2146
positive,887


### Policy `minimal-v1` (контракт для D2-02)

1. Декодировать HTML entities до стабильного результата.
2. Заменить корректные HTML tags/comments пробелами, сохранить текст элементов.
3. Применить `str.lower()`.
4. Свести Unicode whitespace к одному пробелу, убрать пробелы по краям.

Сохраняем отрицания, числа, проценты, валюты и пунктуацию. Не удаляем stopwords,
не выполняем lemmatization и не заменяем uppercase слова на тикеры.
`preprocess_text(str) -> str`: неверный тип вызывает `TypeError`, пустой результат
возвращает `""`. Ожидается идемпотентность; import и вызов не требуют NLTK/network.
Полный контракт и ограничения HTML описаны в SPEC §7.

Пример **ожидаемого**, ещё не исполненного преобразования:
`Profit is NOT down -5% &amp; costs £2.50.` →
`profit is not down -5% & costs £2.50.`

### Будущая схема и исключения

Canonical `(N, 2)` → processed `(M, 5)`, `M <= N`:

| Колонка | Значение |
| --- | --- |
| `source_row_id` | 0-based позиция в canonical CSV до фильтрации |
| `text` | Исходный текст |
| `sentiment` | Исходная строковая метка |
| `text_clean` | Непустой очищенный текст |
| `label_id` | `0=negative`, `1=neutral`, `2=positive` |

Неверная схема, тип или неизвестный класс — явные ошибки.
Missing text/label, whitespace-only и пустой результат очистки — исключения
с source IDs и причинами. Строка учитывается один раз, даже при нескольких причинах.
Исходные данные и порядок сохраняются. Дубликаты пока сохраняются, conflicts
потребуют явного решения; в Day03 совпадающие raw/clean texts группируются при split.
Эти поля и отчёты **ещё не создаются** на checkpoint D2-01.

In [3]:
pd.testing.assert_frame_equal(raw_news, raw_snapshot)
pd.testing.assert_frame_equal(raw_news, load_raw_news(CANONICAL_PATH))
canonical_sha256_after = hashlib.sha256(CANONICAL_PATH.read_bytes()).hexdigest()
assert canonical_sha256_before == canonical_sha256_after
assert tuple(raw_news.columns) == ("text", "sentiment")
print("Verified: DataFrame and canonical file unchanged; repeated load preserves rows/order.")
print("D2-01 complete: policy and raw-data setup; preprocessing implementation starts in D2-02.")

Verified: DataFrame and canonical file unchanged; repeated load preserves rows/order.
D2-01 complete: policy and raw-data setup; preprocessing implementation starts in D2-02.


### Интерпретация и ограничения

Вывод shape/counts относится к реальному локальному canonical CSV.
Повторная загрузка совпала с исходной таблицей; checksum файла не изменился.
Это не доказывает корректность будущей очистки: её проверки появятся в D2-02.
Числовые class IDs — категории, а не шкала интенсивности тональности.
Lowercase теряет регистр; минимальная HTML policy может принять `<USD>` за tag.
Рост macro F1 не обещаем: модель ещё не обучалась.

### Самопроверка

- Как удаление `not` и `-5%` может изменить смысл предложения?
- Почему `None` нельзя незаметно превращать в строку `"None"`?
- Зачем source ID назначается до фильтрации, а не после неё?
- Почему очистка не требует `fit`, а TF-IDF требует train?
- Что проверено на D2-01, а что остаётся проверить в D2-02?

### Повторный запуск

Выберите kernel **Python (Text-classification venv)**, затем Restart Kernel → Run All.
Из корня: `venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=text-classification notebooks/day02_preprocessing.ipynb`.
Из `notebooks/`: `../venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=text-classification day02_preprocessing.ipynb`.
Типы всех code cells: `venv/bin/python scripts/check_types.py` из корня.

## D2-02 — очистка одного текста

**Цель:** реализовать `preprocess_text(str) -> str` по policy `minimal-v1`.

**Краткая теория.** Здесь нет обучаемых параметров: одинаковый текст и policy
дают одинаковый результат без `fit`. Декодируем entities, заменяем tags/comments
пробелами, нормализуем регистр и whitespace. Повторяем до стабильного результата
для вложенного encoding и идемпотентности. Отрицания, суммы, проценты и пунктуация
сохраняются. NLTK resources и network не нужны.

**Вызов кода:** возьмём первое реальное предложение каждого класса в порядке
`negative`, `neutral`, `positive`. Source position показываем только для проверки
примеров, processed dataset и его source IDs появятся в D2-03.

In [4]:
from finnews_sentiment.features.preprocess import PREPROCESSING_POLICY, preprocess_text

example_rows: list[dict[str, str | int]] = []
for sentiment in CLASS_ORDER:
    source_position = next(
        position for position, value in enumerate(raw_news["sentiment"])
        if value == sentiment
    )
    raw_value = raw_news.iloc[source_position]["text"]
    if not isinstance(raw_value, str):
        raise TypeError("Real example text must be str")
    cleaned_value = preprocess_text(raw_value)
    assert preprocess_text(cleaned_value) == cleaned_value
    example_rows.append({
        "source_position": source_position,
        "sentiment": sentiment,
        "text": raw_value,
        "text_clean": cleaned_value,
    })

print(f"Preprocessing policy: {PREPROCESSING_POLICY}")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(example_rows))
for example in example_rows:
    print(f"\n{example['sentiment']} (source position {example['source_position']})")
    print(f"raw:   {example['text']}")
    print(f"clean: {example['text_clean']}")

pd.testing.assert_frame_equal(raw_news, raw_snapshot)
assert hashlib.sha256(CANONICAL_PATH.read_bytes()).hexdigest() == canonical_sha256_before
print("Verified: examples are idempotent; source DataFrame and CSV remain unchanged.")

Preprocessing policy: minimal-v1


,source_position,sentiment,text,text_clean
0,393,negative,"Jan. 6 -- Ford is struggling in the face of slowing truck and SUV sales and a surfeit of up-to-date , gotta-have cars .","jan. 6 -- ford is struggling in the face of slowing truck and suv sales and a surfeit of up-to-date , gotta-have cars ."
1,0,neutral,"According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .","according to gran , the company has no plans to move all production to russia , although that is where the company is growing ."
2,1,positive,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,with the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .



negative (source position 393)
raw:   Jan. 6 -- Ford is struggling in the face of slowing truck and SUV sales and a surfeit of up-to-date , gotta-have cars .
clean: jan. 6 -- ford is struggling in the face of slowing truck and suv sales and a surfeit of up-to-date , gotta-have cars .

neutral (source position 0)
raw:   According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .
clean: according to gran , the company has no plans to move all production to russia , although that is where the company is growing .

positive (source position 1)
raw:   With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .
clean: with the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefo

### Интерпретация

В реальных примерах регистр нормализуется, внешние и повторные пробелы убираются
при наличии. Числа, десятичные точки, `%`, скобки и слова сохраняются.
В neutral-примере сохранено отрицание `no plans`; в negative-примере
сохранены `Jan. 6`, `--` и дефисы, а `SUV` становится `suv`. Метки не меняются.
Эти три примера демонстрируют lowercase; валюты и проценты покрыты offline tests.
Если в этих предложениях нет HTML, они не подтверждают HTML boundary cases:
entities, tags, comments, сравнения и неверные типы проверяются отдельными offline tests.
Идемпотентность трёх примеров и сохранность исходника подтверждены assertions.

**Ограничения:** lowercase теряет регистр, `<USD>` может считаться tag;
это cleanup предложений, не HTML scraping. Нет измерения macro F1.
Табличные helpers в модуле остаются заготовками: текущий checkpoint вызывает
только `preprocess_text`. Исключения строк, mapping и duplicate policy — следующие шаги.

### Самопроверка

- Почему здесь нет `fit`, vocabulary и обучаемых параметров?
- Что произойдёт с `None`, пустой строкой и строкой из HTML tags?
- Почему удаление цифр, валют или `not` сейчас не входит в policy?
- Зачем проверять `preprocess_text(preprocess_text(s)) == preprocess_text(s)`?
- Почему три реальных примера не заменяют tests на HTML и неверные типы?